In [5]:
import pandas as pd


In [6]:
import pandas as pd
import numpy as np

# Task 1: Dataset Selection and Reading
spotify_df = pd.read_csv('Spotify.csv')
print("Dataset Shape:", spotify_df.shape)
print("\nFirst 5 Rows:")
display(spotify_df.head())

Dataset Shape: (37000, 10)

First 5 Rows:


,ids,names,popularity,followers,genres,first_release,last_release,num_releases,num_tracks,monthly_listeners
0,68JRsR0O5ynf8xhu2jk2no,eqd,6,1974,NaN,2007,2023,10,2,1444.0
1,5VvZ6Br2O5ffnhkZt2HE8Q,チキンガーリックステーキ,3,140,NaN,2004,2010,12,2,231.0
2,2P3aLwJ41aRYkyFFw2FRM7,sam concepcion,45,168497,opm,2007,2023,20,1,583900.0
3,3ZzXi1Y1BvnvKJSVvzU2iP,vibemares,0,151,NaN,1996,1997,2,5,101.0
4,2ZC9B5qSrV48WOrOujZkNC,lantinor,0,1,NaN,2019,2019,1,12,0.0


In [7]:
# Task 2: Data Understanding
print("--- Statistical Summary ---")
display(spotify_df.describe())

print("\n--- Data Structure & Types ---")
spotify_df.info()

print("\n--- Missing Values Count ---")
print(spotify_df.isnull().sum())

--- Statistical Summary ---


,popularity,followers,first_release,last_release,num_releases,num_tracks,monthly_listeners
count,37000.000000,3.700000e+04,37000.000000,37000.000000,37000.000000,37000.000000,3.698800e+04
mean,9.537486,3.966586e+04,1678.249162,1685.222027,6.124216,5.058514,1.096624e+05
std,14.112742,7.126790e+05,745.692094,748.741395,6.688222,11.699609,1.246455e+06
min,0.000000,0.000000e+00,-1.000000,-1.000000,0.000000,0.000000,0.000000e+00
25%,0.000000,2.900000e+01,1994.000000,2008.000000,1.000000,1.000000,2.000000e+01
50%,2.000000,2.020000e+02,2010.000000,2019.000000,3.000000,2.000000,2.630000e+02
75%,15.000000,1.726000e+03,2017.000000,2023.000000,9.000000,6.000000,5.072500e+03
max,92.000000,8.726594e+07,2024.000000,2024.000000,20.000000,810.000000,8.135858e+07



--- Data Structure & Types ---
<class 'pandas.DataFrame'>
RangeIndex: 37000 entries, 0 to 36999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ids                37000 non-null  str    
 1   names              37000 non-null  str    
 2   popularity         37000 non-null  int64  
 3   followers          37000 non-null  int64  
 4   genres             12437 non-null  str    
 5   first_release      37000 non-null  int64  
 6   last_release       37000 non-null  int64  
 7   num_releases       37000 non-null  int64  
 8   num_tracks         37000 non-null  int64  
 9   monthly_listeners  36988 non-null  float64
dtypes: float64(1), int64(6), str(3)
memory usage: 2.8 MB

--- Missing Values Count ---
ids                      0
names                    0
popularity               0
followers                0
genres               24563
first_release            0
last_release             0
num_releases     

In [8]:
# Task 3: Data Transformation
# Apply track-level transformations only when the loaded dataset provides those columns.
if 'explicit' in spotify_df.columns:
    spotify_df['is_explicit_flag'] = (
        spotify_df['explicit'].astype('string').str.strip().str.casefold() == 'yes'
    ).astype(int)
else:
    print("Skipping explicit flag: the dataset has no 'explicit' column.")

if 'energy' in spotify_df.columns:
    spotify_df['energy_level'] = pd.qcut(
        spotify_df['energy'],
        q=3,
        labels=['Low Energy', 'Medium Energy', 'High Energy'],
        duplicates='drop'
    )
else:
    print("Skipping energy levels: the dataset has no 'energy' column.")

print("Transformed Columns Sample:")
display(spotify_df[['ids', 'names', 'popularity', 'followers']].head())

Skipping explicit flag: the dataset has no 'explicit' column.
Skipping energy levels: the dataset has no 'energy' column.
Transformed Columns Sample:


,ids,names,popularity,followers
0,68JRsR0O5ynf8xhu2jk2no,eqd,6,1974
1,5VvZ6Br2O5ffnhkZt2HE8Q,チキンガーリックステーキ,3,140
2,2P3aLwJ41aRYkyFFw2FRM7,sam concepcion,45,168497
3,3ZzXi1Y1BvnvKJSVvzU2iP,vibemares,0,151
4,2ZC9B5qSrV48WOrOujZkNC,lantinor,0,1


In [13]:
# Task 4: Handling Null Values using Mean Imputation
numeric_columns = spotify_df.select_dtypes(include='number').columns
columns_to_impute = [
    column for column in numeric_columns
    if spotify_df[column].isnull().any()
]

print("Missing numeric values before imputation:")
print(spotify_df[columns_to_impute].isnull().sum())

# Fill missing numeric values using each column's mean.
for column in columns_to_impute:
    spotify_df[column] = spotify_df[column].fillna(spotify_df[column].mean())

print("\nMissing numeric values after mean imputation:")
print(spotify_df[columns_to_impute].isnull().sum())

Missing numeric values before imputation:
Series([], dtype: float64)

Missing numeric values after mean imputation:
Series([], dtype: float64)


In [15]:
# Task 5: Insight Generation

genre_summary = spotify_df.assign(
    genres=spotify_df["genres"].fillna("Unknown")
).groupby("genres").agg({
    "popularity": "mean",
    "followers": "sum",
    "num_tracks": "sum",
    "monthly_listeners": "sum"
}).rename(columns={
    "popularity": "average_popularity",
    "followers": "total_followers",
    "num_tracks": "total_tracks",
    "monthly_listeners": "total_monthly_listeners"
}).reset_index()

print("--- Genre-Wise Analysis ---")
display(genre_summary.sort_values("total_followers", ascending=False))

artist_summary = spotify_df.groupby("names").agg({
    "followers": "sum",
    "popularity": "mean",
    "num_tracks": "sum",
    "monthly_listeners": "sum"
}).rename(columns={
    "followers": "total_followers",
    "popularity": "average_popularity",
    "num_tracks": "total_tracks",
    "monthly_listeners": "total_monthly_listeners"
}).reset_index()

print("\n--- Top Performing Artists ---")
display(artist_summary.sort_values("total_followers", ascending=False))

--- Genre-Wise Analysis ---


,genres,average_popularity,total_followers,total_tracks,total_monthly_listeners
6215,pop,75.285714,109254337,79,2.618642e+08
1618,"canadian hip hop, canadian pop, hip hop, pop r...",92.000000,87265940,1,8.135858e+07
2207,"classic rock, glam rock, rock",81.000000,49487244,16,5.178721e+07
5214,"latin pop, reggaeton, reggaeton colombiano, tr...",85.000000,33513527,1,4.310513e+07
23,Unknown,4.612710,28967077,119851,2.663952e+08
...,...,...,...,...,...
4419,historic string quartet,0.000000,3,0,2.000000e+00
3228,dunedin indie,0.000000,2,1,1.000000e+00
1688,cante alentejano,0.000000,2,0,1.300000e+01
485,appenzeller folk,0.000000,2,0,1.900000e+01



--- Top Performing Artists ---


,names,total_followers,average_popularity,total_tracks,total_monthly_listeners
8937,drake,87265940,92.0,1,81358580.0
25674,queen,49487244,81.0,16,51787211.0
23587,olivia rodrigo,34059740,88.0,4,58313600.0
19814,maluma,33513527,85.0,1,43105132.0
16962,katy perry,32842230,81.0,6,55922564.0
...,...,...,...,...,...
12,1 a.m.,0,0.0,0,2.0
16425,judz,0,0.0,0,0.0
62,21gramm,0,0.0,1,0.0
56,20 hz,0,0.0,0,1.0
